In [29]:
import math

DATA = {
    "requests": [
        [ 3738.682118759545,   3176.3620564508888,  0.9255958108248765, 1, 1358.7472330159917, 1242.4073038422487, 1602.4073038422487],
        [-7332.379353375232,  -4188.522793367846,   0.7478413173478692, 1,  605.3789606058994,  652.9093445397065, 1012.9093445397065],
        [ 5777.475570492262,  -7986.273425724185,   0.9634110342579557, 1, 1102.7598350528779, 1237.7513484295284, 1597.7513484295284],
        [ 6996.942373406358,   4418.535163659321,   0.9908166528639936, 1,  722.3235983813729,  969.8366216562558, 1329.8366216562558],
        [ 4400.906550671561,   2432.3349996395264, 10.776638065660618,  0, 1149.8366216562558, 1178.7472330159917, 1538.7472330159917],
        [-1169.2698685472992,  3497.237946688962,  27.380202204619298,  0,    0.0,               55.679027962466364,  415.67902796246636],
        [ 6710.724138902582,  -3348.9561871222304,  0.2506526123531701, 1,    0.0,               59.670309697854066,  419.67030969785407],
        [ -216.72058856320808,-3938.133905039083,   0.8925640200120823, 1,  479.34061939570813, 425.37896060589935,  785.3789606058994 ],
        [-3536.845602701436,   6049.129314355786,   1.0952132256820593, 1,  235.67902796246636, 278.16018765158515,  638.1601876515851 ],
        [  586.8106635735363,  6329.938959833223,   0.5507433758448146, 1,  458.16018765158515, 542.3235983813729,   902.3235983813729 ],
    ],
    "truck_vel": 15.6464,
    "drone_vel": 31.2928,
    "truck_cap": 400.0,
    "drone_cap": 2.27,
    "drone_lim": 700.0,
    "truck_num": 1,
    "drone_num": 1,
}

ALPHA = 1.0
GAMMA = 1000.0
LW   = 3600.0
DEPOT = [0.0, 0.0]

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
#  KHOẢNG CÁCH
# ─────────────────────────────────────────────────────────────────────────────

def dist(a, b):
    """Khoảng cách Euclidean giữa hai điểm a và b (đơn vị: mét)."""
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)


# ─────────────────────────────────────────────────────────────────────────────
#  ĐÁNH GIÁ MỘT TRIP
# ─────────────────────────────────────────────────────────────────────────────

def eval_trip(stops, depart, vel, cap, dist_lim, requests, Lw):
    """
    Mô phỏng chuyến đi theo thứ tự `stops` và kiểm tra tất cả ràng buộc.

    Luồng của một trip:
        depot ──► stop[0] ──► stop[1] ──► ... ──► stop[n] ──► depot
          t=depart   t=p_0      t=p_1                t=p_n      t=q_depot

    Trả về dict kết quả nếu trip hợp lệ, hoặc None nếu vi phạm ràng buộc.
    """

    pos        = DEPOT[:]  # vị trí hiện tại của xe/drone, bắt đầu tại depot
    t          = depart    # đồng hồ thời gian, tính từ lúc xuất phát (giây)
    total_dist = 0.0       # tổng quãng đường đã đi trong trip này (mét)
    total_load = 0.0       # tổng khối lượng hàng đã nhận (kg)
    pis        = {}        # lưu pickup time của từng stop: {req_index: p_i}

    for ri in stops:
        x, y, dem, _, r_i, e_i, l_i = requests[ri]
        # x, y : tọa độ khách hàng
        # dem  : khối lượng hàng tại stop này (kg)
        # e_i  : earliest time — cửa sổ mở, đến trước thì phải chờ đến e_i
        # l_i  : latest time  — cửa sổ đóng, đến sau l_i là vi phạm

        # ── Kiểm tra tải trọng ───────────────────────────────────────────
        total_load += dem
        if total_load > cap:
            # Tổng hàng vượt quá sức chứa của xe/drone → trip không hợp lệ
            return None

        # ── Di chuyển đến stop ───────────────────────────────────────────
        leg = dist(pos, [x, y])  # quãng đường từ vị trí hiện tại đến stop này
        total_dist += leg
        if total_dist > dist_lim:
            # Tổng quãng đường vượt tầm bay tối đa
            # (chỉ áp dụng với drone; truck có dist_lim = inf)
            return None

        # ── Tính thời điểm đến và pickup ─────────────────────────────────
        a_i = t + leg / vel
        # a_i : arrival time — thời điểm xe/drone đến nơi khách (giây)
        #       = thời điểm xuất phát khỏi stop trước + thời gian di chuyển

        if a_i > l_i:
            # Đến sau khi cửa sổ thời gian đã đóng → vi phạm time window
            return None

        p_i = max(a_i, e_i)
        # p_i : pickup time — thời điểm thực sự lấy hàng (giây)
        #   Nếu đến trước e_i  → xe phải CHỜ đến e_i rồi mới lấy hàng → p_i = e_i
        #   Nếu đến sau  e_i   → lấy hàng ngay lúc đến               → p_i = a_i
        # Ví dụ: a_i=50s, e_i=100s → p_i=100s (chờ 50 giây)
        #        a_i=150s, e_i=100s → p_i=150s (đến trễ hơn, lấy luôn)

        pis[ri] = p_i   # lưu lại p_i để tính W_i sau khi biết q_depot
        pos = [x, y]    # cập nhật vị trí hiện tại → xe đang ở chỗ khách
        t   = p_i       # cập nhật đồng hồ → tiếp tục đi từ thời điểm p_i

    # ── Quay về depot ────────────────────────────────────────────────────
    back = dist(pos, DEPOT)  # quãng đường từ stop cuối về depot
    total_dist += back
    if total_dist > dist_lim:
        # Drone hết tầm bay trước khi về đến depot
        return None

    q_depot = t + back / vel
    # q_depot : thời điểm xe/drone về đến depot (giây)
    #           Đây là "q" trong công thức W_i = q_depot - p_i
    #           Tất cả hàng trên xe đều được coi là "giao xong" tại thời điểm này

    # ── Kiểm tra waiting time cho mọi stop ──────────────────────────────
    wis = {}
    for ri, p_i in pis.items():
        W_i = q_depot - p_i
        # W_i : waiting time — thời gian hàng nằm trên xe/drone (giây)
        #       = từ lúc lấy hàng tại khách (p_i) đến lúc xe về depot (q_depot)
        #       Stop đứng càng đầu route → W_i càng lớn (phải chờ xe đi hết các stop sau)
        if W_i > Lw:
            # Hàng "ngồi" trên xe quá lâu, vượt giới hạn Lw (= 3600s = 60 phút)
            return None
        wis[ri] = W_i

    # Tất cả ràng buộc đều thỏa → trả về kết quả
    return {
        "q_depot":    q_depot,     # thời điểm về depot (giây)
        "total_dist": total_dist,  # tổng quãng đường của trip (mét)
        "pis":        pis,         # {req_index: p_i}  pickup time từng stop
        "wis":        wis,         # {req_index: W_i}  waiting time từng stop
        "load":       total_load,  # tổng khối lượng hàng trong trip (kg)
    }


# ─────────────────────────────────────────────────────────────────────────────
#  TÌM VỊ TRÍ CHÈN TỐT NHẤT
# ─────────────────────────────────────────────────────────────────────────────

def best_insert(current_stops, new_req, avail, vel, cap, dist_lim, requests, Lw):
    """
    Thử chèn new_req vào mọi vị trí trong current_stops,
    chọn vị trí làm tăng tổng quãng đường ít nhất (Δdist nhỏ nhất).

    Ví dụ current_stops = [5, 8], new_req = 9:
        Thử vị trí 0: [9, 5, 8]
        Thử vị trí 1: [5, 9, 8]  ← giả sử vị trí này tốt nhất
        Thử vị trí 2: [5, 8, 9]

    Trả về (new_stops, result, delta) nếu tìm được vị trí hợp lệ,
    hoặc None nếu không vị trí nào feasible.
    """

    # Tính quãng đường trip hiện tại (trước khi thêm new_req) để tính Δdist
    base_res  = eval_trip(current_stops, avail, vel, cap, dist_lim, requests, Lw)
    base_dist = base_res["total_dist"] if base_res else 0.0
    # Nếu trip hiện tại rỗng (stops=[]) thì base_dist = 0

    best = None  # sẽ lưu (new_stops, eval_result, delta) tốt nhất tìm được

    for pos in range(len(current_stops) + 1):
        # Thử chèn new_req vào vị trí `pos`:
        #   current_stops[:pos]  = các stop trước vị trí chèn
        #   [new_req]            = stop mới
        #   current_stops[pos:]  = các stop sau vị trí chèn
        candidate = current_stops[:pos] + [new_req] + current_stops[pos:]

        res = eval_trip(candidate, avail, vel, cap, dist_lim, requests, Lw)
        if res is None:
            # Vị trí này vi phạm ràng buộc → bỏ qua, thử vị trí khác
            continue

        delta = res["total_dist"] - base_dist
        # delta : quãng đường tăng thêm khi chèn new_req vào vị trí này
        #         delta nhỏ = ít tốn thêm đường = tốt hơn

        if best is None or delta < best[2]:
            best = (candidate, res, delta)  # cập nhật lựa chọn tốt nhất

    return best  # None nếu không có vị trí nào feasible


# ─────────────────────────────────────────────────────────────────────────────
#  SOLVER CHÍNH
# ─────────────────────────────────────────────────────────────────────────────

def solve(data, Lw, alpha, gamma):
    """
    Thuật toán tham lam giải bài toán giao hàng động.

    Ý tưởng:
        Requests xuất hiện dần theo thời gian (xử lý theo thứ tự r_i).
        Với mỗi request mới:
          Bước 1 — Thử chèn vào trip đang mở (ưu tiên drone, sau đó truck).
          Bước 2 — Nếu không chèn được: dispatch trip cũ, mở trip mới.
          Bước 3 — Nếu vẫn không được: reject request đó.
        Cuối cùng: dispatch tất cả trip còn dang dở.
    """

    # ── Đọc tham số hệ thống ─────────────────────────────────────────────
    t_vel    = data["truck_vel"]         # vận tốc xe tải (m/s)
    d_vel    = data["drone_vel"]         # vận tốc drone (m/s)
    t_cap    = data["truck_cap"]         # tải trọng tối đa xe tải (kg)
    d_cap    = data["drone_cap"]         # tải trọng tối đa drone (kg)
    d_lim    = data["drone_lim"] * d_vel # drone_lim gốc là endurance (giây)
                                         # → nhân với vận tốc = giới hạn khoảng cách (mét)
                                         # ví dụ: 700s × 31.29m/s = 21,905m
    n_trucks = data["truck_num"]         # số lượng xe tải
    n_drones = data["drone_num"]         # số lượng drone
    requests = data["requests"]          # danh sách tất cả requests

    # ── Trạng thái sẵn sàng của từng xe/drone ────────────────────────────
    # Giá trị = thời điểm (giây) xe/drone đó về đến depot và sẵn sàng cho trip tiếp theo.
    # Ban đầu = 0 vì tất cả đang đứng ở depot, chưa đi đâu.
    # Sau khi dispatch trip, giá trị này = q_depot của trip vừa dispatch.
    truck_avail = [0.0] * n_trucks   # ví dụ: [0.0] nếu có 1 xe tải
    drone_avail = [0.0] * n_drones   # ví dụ: [0.0] nếu có 1 drone

    # ── Open trips ────────────────────────────────────────────────────────
    # "Open trip" = chuyến đang được xây dựng, chưa chốt (chưa dispatch).
    # Mỗi xe/drone có tối đa 1 open trip tại 1 thời điểm.
    # Giá trị: None = không có trip nào đang mở
    #          list = danh sách index request dự kiến phục vụ, ví dụ: [5, 8, 9]
    open_truck = [None] * n_trucks
    open_drone = [None] * n_drones

    # ── Biến lưu kết quả ─────────────────────────────────────────────────
    dispatched_trips = []  # danh sách tất cả trip đã dispatch (dict thông tin chi tiết)
    served           = set()  # tập index các request đã được phục vụ thành công
    unserved         = []     # danh sách index các request bị từ chối
    dist_truck       = 0.0    # tổng quãng đường xe tải đã đi (mét)
    dist_drone       = 0.0    # tổng quãng đường drone đã bay (mét)
    trip_id          = [0]    # bộ đếm trip, dùng list vì cần thay đổi trong hàm lồng
                              # (Python không cho gán lại biến ngoài trong nested function
                              #  nếu dùng biến thường, nhưng có thể mutate list)

    # ════════════════════════════════════════════════════════════════════
    #  HÀM LỒNG: DISPATCH
    # ════════════════════════════════════════════════════════════════════
    def dispatch(vtype, vidx, stops, avail):
        """
        Xác nhận (chốt) một open trip: tính toán lịch trình và lưu kết quả.

        Tham số:
            vtype : "truck" hoặc "drone"
            vidx  : index của xe/drone trong danh sách (0, 1, 2, ...)
            stops : list index request trong trip này, ví dụ [5, 8, 9]
            avail : thời điểm xe/drone sẵn sàng xuất phát từ depot (giây)

        Trả về:
            q_depot nếu trip feasible (xe sẽ về depot lúc này)
            avail   nếu trip không feasible (xe không đi, vẫn ở depot)
        """
        nonlocal dist_truck, dist_drone  # cần ghi vào biến của hàm cha
        trip_id[0] += 1                  # tăng bộ đếm trip

        # Chọn tham số phù hợp với loại xe
        vel  = d_vel if vtype == "drone" else t_vel
        cap  = d_cap if vtype == "drone" else t_cap
        dlim = d_lim if vtype == "drone" else float("inf")
        # Truck không có giới hạn khoảng cách → dùng inf

        # Kiểm tra lần cuối trước khi chốt
        res = eval_trip(stops, avail, vel, cap, dlim, requests, Lw)

        if res is None:
            # Trip không feasible tại thời điểm dispatch
            # (hiếm xảy ra vì đã kiểm tra khi insert, nhưng vẫn cần xử lý)
            print(f"  [WARN] Trip {vtype.upper()}-{vidx} không feasible → reject {stops}")
            for ri in stops:
                unserved.append(ri)
            return avail   # xe không đi, vẫn sẵn sàng tại depot từ lúc `avail`

        # Trip hợp lệ → ghi nhận kết quả
        label = f"{vtype.capitalize()}-{vidx} trip#{trip_id[0]}"
        print(f"  ✓ DISPATCH {label:25s} stops={stops}  "
              f"depart={avail:.0f}s  q_depot={res['q_depot']:.0f}s  "
              f"dist={res['total_dist']:.0f}m  load={res['load']:.2f}kg")

        dispatched_trips.append({
            "label":      label,
            "vtype":      vtype,
            "stops":      stops,
            "depart":     round(avail, 2),           # thời điểm xuất phát từ depot
            "q_depot":    round(res["q_depot"], 2),  # thời điểm về depot
            "total_dist": round(res["total_dist"], 2),
            "load":       round(res["load"], 3),
            "pis": {k: round(v, 2) for k, v in res["pis"].items()},  # pickup time từng stop
            "wis": {k: round(v, 2) for k, v in res["wis"].items()},  # waiting time từng stop
        })

        # Cộng dồn quãng đường vào đúng loại xe
        if vtype == "drone":
            dist_drone += res["total_dist"]
        else:
            dist_truck += res["total_dist"]

        # Đánh dấu các request trong trip này là đã được phục vụ
        for ri in stops:
            served.add(ri)

        # Trả về thời điểm xe/drone về đến depot → dùng để cập nhật avail
        return res["q_depot"]

    # ════════════════════════════════════════════════════════════════════
    #  VÒNG LẶP CHÍNH: XỬ LÝ TỪNG REQUEST
    # ════════════════════════════════════════════════════════════════════

    # Sắp xếp requests theo r_i (thời điểm xuất hiện) tăng dần
    # enumerate() để giữ lại index gốc (orig_idx) sau khi sort
    sorted_reqs = sorted(enumerate(requests), key=lambda kv: kv[1][4])

    for orig_idx, req in sorted_reqs:
        x, y, demand, can_drone, r_i, e_i, l_i = req
        # orig_idx  : index gốc của request trong danh sách requests[]
        # can_drone : 1 = drone được phép phục vụ, 0 = chỉ truck
        # r_i       : thời điểm request này xuất hiện (giây)
        assigned = False  # cờ đánh dấu request này đã được gán cho xe nào chưa

        print(f"\n[Req #{orig_idx}]  r_i={r_i:.0f}s  "
              f"window=[{e_i:.0f},{l_i:.0f}]s  "
              f"demand={demand:.3f}kg  drone={'Y' if can_drone else 'N'}")

        # ════════════════════════════════════════════════════════════════
        #  BƯỚC 1: THỬ CHÈN VÀO OPEN TRIP ĐANG MỞ
        # ════════════════════════════════════════════════════════════════
        # Tìm tất cả open trips có thể nhận thêm request này,
        # sau đó chọn open trip nào mà việc chèn tốn ít đường nhất (Δdist nhỏ nhất).

        # Gom danh sách các "ứng viên" — open trips đang mở và phù hợp:
        # mỗi phần tử là (loại xe, index xe, danh sách stops hiện tại, thời điểm avail)
        candidates = []

        # Ưu tiên drone trước (nhanh hơn, tiết kiệm truck cho hàng nặng)
        if can_drone and demand <= d_cap:
            # can_drone=1 và hàng không vượt tải drone mới được xét
            for dk in range(n_drones):
                if open_drone[dk] is not None:
                    # Drone dk đang có open trip → thêm vào danh sách ứng viên
                    candidates.append(("drone", dk, open_drone[dk], drone_avail[dk]))

        # Sau đó xét truck
        for tk in range(n_trucks):
            if open_truck[tk] is not None:
                candidates.append(("truck", tk, open_truck[tk], truck_avail[tk]))

        # Tìm ứng viên tốt nhất: thử best_insert trên từng open trip,
        # chọn cái có Δdist nhỏ nhất
        best_option = None   # sẽ lưu (delta, vtype, vidx, new_stops)

        for vtype, vidx, stops, avail in candidates:
            vel  = d_vel if vtype == "drone" else t_vel
            cap  = d_cap if vtype == "drone" else t_cap
            dlim = d_lim if vtype == "drone" else float("inf")

            ins = best_insert(stops, orig_idx, avail, vel, cap, dlim, requests, Lw)
            if ins is None:
                continue  # không chèn được vào trip này → thử trip khác

            new_stops, res, delta = ins
            # new_stops : danh sách stops mới sau khi chèn
            # delta     : quãng đường tăng thêm

            if best_option is None or delta < best_option[0]:
                best_option = (delta, vtype, vidx, new_stops)

        if best_option is not None:
            delta, vtype, vidx, new_stops = best_option

            # Cập nhật open trip với stops mới (đã có request vừa chèn vào)
            if vtype == "drone":
                open_drone[vidx] = new_stops
            else:
                open_truck[vidx] = new_stops

            assigned = True
            print(f"  → INSERT vào {vtype.upper()}-{vidx}: {new_stops}  "
                  f"Δdist={delta:.0f}m")

        # ════════════════════════════════════════════════════════════════
        #  BƯỚC 2: MỞ TRIP MỚI NẾU KHÔNG CHÈN ĐƯỢC
        # ════════════════════════════════════════════════════════════════
        # Không có open trip nào chấp nhận request này.
        # → Phải tạo trip mới, nhưng mỗi xe chỉ có 1 open trip tại 1 lúc.
        # → Nếu xe đang có open trip thì phải dispatch trip đó trước,
        #   giải phóng xe để mở trip mới.

        if not assigned:

            # Thử drone trước (ưu tiên giữ truck cho hàng nặng)
            if can_drone and demand <= d_cap:
                for dk in range(n_drones):

                    if open_drone[dk] is not None:
                        # Drone đang bận với trip cũ → dispatch trip cũ để giải phóng
                        drone_avail[dk] = dispatch("drone", dk,
                                                    open_drone[dk], drone_avail[dk])
                        open_drone[dk] = None
                        # Sau dispatch, drone_avail[dk] = q_depot của trip cũ
                        # nghĩa là drone chỉ rảnh sau khi về depot từ trip cũ

                    # Kiểm tra xem request này (đứng một mình) có feasible không
                    # với drone vừa được giải phóng (hoặc đang rảnh từ đầu)
                    res = eval_trip([orig_idx], drone_avail[dk],
                                    d_vel, d_cap, d_lim, requests, Lw)
                    if res is not None:
                        # Feasible → mở trip mới chứa mỗi mình request này
                        open_drone[dk] = [orig_idx]
                        assigned = True
                        print(f"  → NEW DRONE-{dk} trip: [{orig_idx}]")
                        break  # dùng drone đầu tiên rảnh là đủ

            # Fallback: thử truck nếu drone không nhận được
            if not assigned and demand <= t_cap:
                for tk in range(n_trucks):

                    if open_truck[tk] is not None:
                        # Truck đang bận → dispatch trip cũ trước
                        truck_avail[tk] = dispatch("truck", tk,
                                                    open_truck[tk], truck_avail[tk])
                        open_truck[tk] = None

                    # Kiểm tra request này đứng một mình có feasible với truck không
                    res = eval_trip([orig_idx], truck_avail[tk],
                                    t_vel, t_cap, float("inf"), requests, Lw)
                    if res is not None:
                        open_truck[tk] = [orig_idx]
                        assigned = True
                        print(f"  → NEW TRUCK-{tk} trip: [{orig_idx}]")
                        break

        # Không có xe nào nhận được → request bị từ chối
        if not assigned:
            unserved.append(orig_idx)
            print(f"  ✗ REJECT req#{orig_idx}")

    # ════════════════════════════════════════════════════════════════════
    #  BƯỚC 3: DISPATCH TẤT CẢ OPEN TRIPS CÒN LẠI
    # ════════════════════════════════════════════════════════════════════
    # Sau khi xử lý hết requests, các open trips chưa được dispatch
    # (chưa có lý do để dispatch sớm hơn) cần được chốt tại đây.

    print(f"\n{'─'*60}")
    print("Dispatch các open trips còn lại...")

    for tk in range(n_trucks):
        if open_truck[tk]:   # nếu truck tk còn open trip chưa dispatch
            truck_avail[tk] = dispatch("truck", tk, open_truck[tk], truck_avail[tk])
            open_truck[tk]  = None

    for dk in range(n_drones):
        if open_drone[dk]:   # nếu drone dk còn open trip chưa dispatch
            drone_avail[dk] = dispatch("drone", dk, open_drone[dk], drone_avail[dk])
            open_drone[dk]  = None

    # ── Tính các chỉ số kết quả ──────────────────────────────────────────
    n_total  = len(requests)
    n_served = len(served)
    n_reject = n_total - n_served

    makespan = max(max(truck_avail), max(drone_avail))
    # makespan : thời điểm hoàn thành muộn nhất của toàn bộ hệ thống (giây)
    #            = lúc chiếc xe/drone cuối cùng về đến depot

    obj = alpha * dist_truck + alpha * dist_drone + gamma * n_reject
    # obj : giá trị hàm mục tiêu cần tối thiểu hóa
    #       = α × (tổng đường truck) + α × (tổng đường drone) + γ × (số request bị reject)
    #       alpha nhỏ → ưu tiên phục vụ nhiều khách hơn là tiết kiệm đường
    #       gamma lớn → hệ thống cố phục vụ tất cả, kể cả phải đi xa

    return {
        "served":     n_served,               # số request được phục vụ
        "total":      n_total,                # tổng số request
        "rejected":   n_reject,               # số request bị từ chối
        "rate":       round(n_served / n_total * 100, 1),  # tỉ lệ phục vụ (%)
        "dist_truck": round(dist_truck, 1),   # tổng quãng đường xe tải (mét)
        "dist_drone": round(dist_drone, 1),   # tổng quãng đường drone (mét)
        "makespan":   round(makespan, 1),     # thời điểm hoàn thành (giây)
        "objective":  round(obj, 2),          # giá trị hàm mục tiêu
        "trips":      dispatched_trips,       # chi tiết từng trip đã dispatch
        "unserved":   unserved,               # danh sách index request bị reject
    }


# ─────────────────────────────────────────────────────────────────────────────
#  IN KẾT QUẢ
# ─────────────────────────────────────────────────────────────────────────────

def print_result(r):
    W = 65
    print(f"\n{'═'*W}")
    print("  KẾT QUẢ")
    print(f"{'═'*W}")
    print(f"  Phục vụ   : {r['served']}/{r['total']} ({r['rate']}%)")
    if r["unserved"]:
        print(f"  Từ chối   : {r['unserved']}")
    print(f"  Dist Truck : {r['dist_truck']:>10,.1f} m")
    print(f"  Dist Drone : {r['dist_drone']:>10,.1f} m")
    print(f"  Makespan   : {r['makespan']:>10,.1f} s  ({r['makespan']/60:.1f} phút)")
    print(f"  Objective  : α·{r['dist_truck']:,.1f} + α·{r['dist_drone']:,.1f} "
          f"+ γ·{r['rejected']} = {r['objective']:,.2f}")

    print(f"\n  {'Trip':<28} {'Stops':<12} {'Depart':>7} "
          f"{'Return':>7} {'Dist':>8} {'Load':>7}")
    print(f"  {'─'*62}")
    for t in r["trips"]:
        print(f"  {t['label']:<28} {str(t['stops']):<12} "
              f"{t['depart']:>7.0f} {t['q_depot']:>7.0f} "
              f"{t['total_dist']:>8.0f} {t['load']:>7.3f}")
        wi_str = "  W_i: " + ", ".join(
            f"#{k}={v:.0f}s" for k, v in t["wis"].items()
        )
        print(wi_str)
    print(f"{'═'*W}")


# ─────────────────────────────────────────────────────────────────────────────
#  CHẠY
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    result = solve(DATA, Lw=LW, alpha=ALPHA, gamma=GAMMA)
    print_result(result)


[Req #5]  r_i=0s  window=[56,416]s  demand=27.380kg  drone=N
  → NEW TRUCK-0 trip: [5]

[Req #6]  r_i=0s  window=[60,420]s  demand=0.251kg  drone=Y
  → NEW DRONE-0 trip: [6]

[Req #8]  r_i=236s  window=[278,638]s  demand=1.095kg  drone=Y
  → INSERT vào TRUCK-0: [5, 8]  Δdist=6801m

[Req #9]  r_i=458s  window=[542,902]s  demand=0.551kg  drone=Y
  → INSERT vào TRUCK-0: [5, 8, 9]  Δdist=3483m

[Req #7]  r_i=479s  window=[425,785]s  demand=0.893kg  drone=Y
  → INSERT vào DRONE-0: [6, 7]  Δdist=3397m

[Req #1]  r_i=605s  window=[653,1013]s  demand=0.748kg  drone=Y
  ✓ DISPATCH Drone-0 trip#1            stops=[6, 7]  depart=0s  q_depot=588s  dist=18397m  load=1.14kg
  → NEW DRONE-0 trip: [1]

[Req #3]  r_i=722s  window=[970,1330]s  demand=0.991kg  drone=Y
  → INSERT vào TRUCK-0: [5, 8, 9, 3]  Δdist=8607m

[Req #2]  r_i=1103s  window=[1238,1598]s  demand=0.963kg  drone=Y
  ✓ DISPATCH Drone-0 trip#2            stops=[1]  depart=588s  q_depot=1128s  dist=16889m  load=0.75kg
  → NEW DRONE-0 tri